In [ ]:
# packages

import warnings
warnings.filterwarnings('ignore')


import pandas as pd
import numpy as np 
import csv
import spacy_stanza

import os
os.getcwd()

# Retrieve Named Entities

In [ ]:
# load nlp stanza
nlp = spacy_stanza.load_pipeline("nl")

In [ ]:
df = pd.read_csv('NOS_final_topics_v2.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change article_id to integer
df['article_id'] = df['article_id'].astype(int)
df['Text'] = df['Text'].str.replace('[LINE_BREAK]', '\n ')

print(df.shape)
df.head()


In [ ]:
print(df.Text[0])

In [ ]:
df.columns

df.about_covid.value_counts()

# limit to articles about covid
df = df[df.about_covid == 1]
print(df.shape)

In [ ]:
unique_articles_df = df[['article_id', 'Text']].drop_duplicates()

In [ ]:
nlp = spacy_stanza.load_pipeline("nl")


In [ ]:
def tag_text_stanza(text):
    # Create a Sentence object from the text
    doc = nlp(text)
    
    # Get the tagged spans
    spans = doc.ents

    # drop if entity label is not in ['ORG','PER']
    spans = [span for span in spans if span.label_ in ['ORG', 'PER']]
    
    # Create a list of tuples containing the entity text and label
    entities = [(ent.text, ent.label_) for ent in spans]

    # Create an empty dictionary to store unique combinations
    unique_entities_tags = {}

    # Iterate through the list of named entity tag combinations
    for entity, tag in entities:
        # Check if the combination exists in the dictionary
        if (entity, tag) not in unique_entities_tags:
            # If it doesn't exist, add it to the dictionary
            unique_entities_tags[(entity, tag)] = True

    # Convert the keys of the dictionary back into a list
    unique_entities = list(unique_entities_tags.keys())

    return unique_entities

In [ ]:
print(unique_articles_df['Text'].iloc[2])

In [ ]:
# get the first text as test
text = unique_articles_df['Text'].iloc[2]

# tag the text
unique_entities = tag_text_stanza(text)
unique_entities

In [ ]:
unique_articles_df['entities_stanza'] = unique_articles_df['Text'].apply(tag_text_stanza)

In [ ]:
df_exploded = unique_articles_df.explode('entities_stanza')
df_exploded.head()

In [ ]:
print(df_exploded.shape)

In [ ]:
# 2. Split the 'entities' tuples into two separate columns: 'entity_name' and 'entity_type'
df_exploded[['entity_name', 'entity_type']] = pd.DataFrame(df_exploded['entities_stanza'].tolist(), index=df_exploded.index)
print(df_exploded.shape)

In [ ]:
df_exploded.head()

In [ ]:
# create a function to check if Text contains kabinet in its lower case form
def check_kabinet(text):
    if 'kabinet' in text.lower():
        return True
    else:
        return False
    
# if the text contains kabinet, then add entity_name as 'Het kabinet' and entity_type as 'ORG'
df_exploded['kabinet'] = df_exploded['Text'].apply(check_kabinet)

df_exploded['kabinet'].value_counts()

In [ ]:
# get kabinet into a new df
df_kabinet = df_exploded[df_exploded['kabinet'] == True]
# drop entities_stanza and entity_name and entity_type
df_kabinet = df_kabinet.drop(columns = ['entities_stanza', 'entity_name', 'entity_type'])
df_kabinet.head()

df_kabinet['entity_name'] = 'Het kabinet'
df_kabinet['entity_type'] = 'ORG'
df_kabinet.head()

In [ ]:
# concat df_exploded and df_kabinet
df_exploded = pd.concat([df_exploded, df_kabinet], axis = 0)
print(df_exploded.shape)

In [ ]:
df_exploded.head()

# drop kabinet column
df_exploded = df_exploded.drop(columns = 'kabinet')

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('dutch')
print(stopwords)

In [ ]:
df_exploded.isnull().sum()

# see where the null values are
df_exploded[df_exploded['entity_name'].isnull()]

# drop if entity_name is null
df_exploded = df_exploded.dropna(subset = ['entity_name'])

In [ ]:
df_exploded[df_exploded['article_id'] == 2331140]

In [ ]:
import re

# Function to generate name variations (without stopword filtering)
def generate_name_variations(name):
    if not isinstance(name, str):
        return []
    
    # Convert to lowercase and split into parts
    # Keep only alphanumeric words (no symbols or numbers)
    parts = re.findall(r'\b\w+\b', name.lower())
    
    # Remove stopwords from the parts
    parts_filtered = [part for part in parts]
    
    variations = []
    
    if parts_filtered:
        # Use the entire filtered name and the individual parts as variations
        variations.append(' '.join(parts_filtered))  # Add filtered parts as a single variation
        variations.extend(parts_filtered)  # Add individual parts as variations

    return variations

# Create a new column for name variations
df_exploded['name_variations'] = df_exploded['entity_name'].apply(lambda x: generate_name_variations(x))

# Drop from name_variations if instance of name_variations matches stopwords
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if variation not in stopwords])
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if re.match(r'\b\w+\b', variation)])
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if len(variation) > 1])
# drop if variation is number
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if not variation.isnumeric()])



In [ ]:
# see where entity name has punctuation
df_exploded[df_exploded['entity_name'].str.contains(r'[^\w\s]')]


In [ ]:
# Helper function to match whole words
def is_full_word_match(variation, name):
    # Convert both variation and name to lowercase
    variation = variation.lower()
    name = name.lower()
    
    # Use word boundaries to ensure full word matching
    return bool(re.search(r'\b' + re.escape(variation) + r'\b', name))

# Function to map names within each article (case-insensitive, full-word match, and entity type check)
def map_names_within_article(article_df):
    name_mapping = {}
    
    # Step 1: Populate the name mapping with the longest canonical names
    for index, row in article_df.iterrows():
        variations = row['name_variations']
        canonical_name = row['entity_name']
        entity_type = row['entity_type']  # Get the entity type

        for variation in variations:
            variation = variation.lower()  # Make variation lowercase for case-insensitive matching
            
            # Check both variation and entity_type match
            for mapped_name, (current_canonical, current_type) in name_mapping.items():
                if is_full_word_match(variation, mapped_name) or is_full_word_match(mapped_name, variation):
                    if entity_type == current_type:  # Ensure entity types match
                        if len(canonical_name) > len(current_canonical):
                            name_mapping[mapped_name] = (canonical_name, entity_type)
                    break
            else:
                name_mapping[variation] = (canonical_name, entity_type)

    # Step 2: Replace entity names based on the canonical mapping (with full-word and entity-type check)
    article_df['entity_name_new'] = article_df.apply(
        lambda row: next(
            (canonical for variation, (canonical, type_) in name_mapping.items()
             if (is_full_word_match(variation, row['entity_name']) or is_full_word_match(row['entity_name'], variation)) 
             and type_ == row['entity_type']), 
            row['entity_name']
        ), axis=1
    )
    
    return article_df


In [ ]:
df_exploded_test = df_exploded.groupby(['article_id', 'entity_type'], group_keys=False).apply(map_names_within_article)
df_exploded_test.head()

In [ ]:
df_exploded_test.shape

In [ ]:
# see where entity_name and entity_name_new are different
differences = df_exploded_test[df_exploded_test['entity_name'] != df_exploded_test['entity_name_new']]
differences.shape

In [ ]:
# see in differences the entity_type ORG
differences[differences['entity_type'] == 'ORG'][['entity_name', 'entity_name_new']].tail(50)

In [ ]:
differences[differences['entity_type'] == 'PER'][['entity_name', 'entity_name_new']].values

### Note: mapping names of organizations do not work here, only Kamer and WHO makes sense

In [ ]:
# Separate the data into persons (PER) and others (ORG, etc.)
df_persons = df_exploded[df_exploded['entity_type'] == 'PER']
df_others = df_exploded[df_exploded['entity_type'] != 'PER']

# Apply the mapping only to persons
df_persons_updated = df_persons.groupby('article_id', group_keys=False).apply(map_names_within_article)

In [ ]:
# in df_others if the entity_name is Kamer then change it to Tweede Kamer or if the entity_name is WHO then change it to Wereldgezondheidsorganisatie WHO
df_others['entity_name_new'] = df_others['entity_name'].apply(lambda x: 'Tweede Kamer' if x == 'Kamer' else 'Wereldgezondheidsorganisatie WHO' if x == 'WHO' else x)
df_others.head()

In [ ]:
# see where entity type is org the most common entity names
df_others[df_others['entity_type'] == 'ORG']['entity_name'].value_counts().head(50) 

In [ ]:
# see where entity type is org and entity name and entity name new are different
differences = df_others[df_others['entity_name'] != df_others['entity_name_new']]
differences[differences['entity_type'] == 'ORG'][['entity_name', 'entity_name_new']].drop_duplicates()

In [ ]:
# see where entity_name and entity_name_new are different
differences_others = df_others[df_others['entity_name'] != df_others['entity_name_new']]
differences_others

In [ ]:
# Combine the processed persons back with the rest of the data
df_final = pd.concat([df_persons_updated, df_others]).sort_index()

In [ ]:
df_final[df_final['article_id'] == 2331140]

In [ ]:
print(df_final.shape)

In [ ]:
df_final.isnull().sum()

In [ ]:
duplicated = df_final[df_final.duplicated(subset = ['article_id', 'entity_name_new', 'entity_type'], keep = False)]
print(duplicated.shape)
duplicated


In [ ]:
# save duplicated to excel to see 
duplicated.to_excel('coref_resolution/duplicated_entities_fullNOS.xlsx', index = False)

In [ ]:
# drop duplicates keep first
df_final = df_final.drop_duplicates(subset = ['article_id', 'entity_name', 'entity_name_new'], keep = 'first')
print(df_final.shape)

In [ ]:
df_final[df_final['article_id'] == 2331140]

In [ ]:
df_final.groupby(['article_id', 'entity_name_new'])['entity_type'].nunique().value_counts()

In [ ]:
# drop entity_name and change entity_name_new to entity_name
df_final = df_final.drop(columns = ['entity_name'])
df_final.rename(columns = {'entity_name_new': 'entity_name'}, inplace = True)

In [ ]:
print(df_final.shape)

In [ ]:
# Define a function to get the length of the name_variations list
df_final['name_variations_length'] = df_final['name_variations'].apply(len)

# Sort the DataFrame by article_id, entity_name, and the length of name_variations in descending order
df_sorted = df_final.sort_values(by=['article_id', 'entity_name', 'name_variations_length'], ascending=[True, True, False])

# Drop duplicates by keeping the first occurrence, which will be the one with the longest name_variations
df_unique = df_sorted.drop_duplicates(subset=['article_id', 'entity_name'], keep='first')

# Drop the helper column
df_unique = df_unique.drop(columns=['name_variations_length'])

print(df_unique.shape)


In [ ]:
df_unique.head()

In [ ]:
counts_entitytypes = df_unique.groupby(['article_id','entity_name'])['entity_type'].nunique().reset_index()
counts_entitytypes.entity_type.value_counts()

In [ ]:
# see where entity_name includes McDonald

df_unique[df_unique['entity_name'].str.contains('McDonald')]

In [ ]:
df_unique.head()

# sort the df based on article_id, entity_name, and entity_type

df_unique = df_unique.sort_values(['article_id', 'entity_name', 'entity_type'])

In [ ]:
def extract_sentences(text, name_variations):
    sentences = sent_tokenize(text, language='dutch')
    relevant_sentences = [sentence for sentence in sentences if any(name in sentence.lower() for name in name_variations)]
    return relevant_sentences

df_unique['relevant_sentences'] = df_unique.apply(lambda x: extract_sentences(x['Text'], x['name_variations']), axis=1)


In [ ]:
df_unique.isnull().sum()

In [ ]:
df_unique.head()

# make exploded_sentences a string by joining the list of sentences
df_unique['relevant_sentences_string'] = df_unique['relevant_sentences'].apply(lambda x: ' \n'.join(x))
df_unique.relevant_sentences.values[0]

In [ ]:
print(df_unique.relevant_sentences_string.values[0])

In [ ]:
# save the df to csv 
df_unique.to_csv('actor_entities_extracted_fullNOS.csv', index = False, sep=';', quoting=csv.QUOTE_NONNUMERIC, encoding = 'utf-8')

# Apply the quote classifier

In [ ]:
# read the df_unique 
df_unique = pd.read_csv('actor_entities_extracted_fullNOS.csv', sep = ';', quoting=csv.QUOTE_NONNUMERIC, encoding = 'utf-8')
print(df_unique.shape)
df_unique.head()

In [ ]:
df_unique.entity_type.value_counts(dropna=False)

In [ ]:
# make a new input text column where you combine entity_name and quoted text with \n
df_unique['input_text'] = df_unique['entity_name'] + '\n' + df_unique['relevant_sentences_string']
df_unique['input_text'].values[:5]

In [ ]:
import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from torch.nn.utils import clip_grad_norm_
import numpy as np
from tqdm import tqdm
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def encode(docs):
    encoded_dict = tokenizer.batch_encode_plus(docs, add_special_tokens=True, padding='max_length',
                                                return_attention_mask=True, truncation=True, return_tensors='pt')
    return encoded_dict['input_ids'], encoded_dict['attention_mask']

tokenizer = AutoTokenizer.from_pretrained("DTAI-KULeuven/robbert-2023-dutch-base")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
huggingface_cache_dir = 'model'

best_model = AutoModelForSequenceClassification.from_pretrained("DTAI-KULeuven/robbert-2023-dutch-base", num_labels=2, cache_dir=huggingface_cache_dir)
best_model.load_state_dict(torch.load('quote_extraction_binary_robbert_best_model.pt'))
best_model = best_model.to(device)

In [ ]:
df_unique['input_text'].values[:5]

In [ ]:
df_unique['input_text'].isnull().sum()

In [ ]:
# where is input text null? 
df_unique[df_unique['input_text'].isnull()]

# drop if input text is null
df_unique = df_unique.dropna(subset = ['input_text'])

In [ ]:
# Tokenize the input text using the encode function
input_ids, attention_mask = encode(df_unique['input_text'].tolist())

# Move inputs to the same device as the model
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Create a dataset and dataloader for batch processing
dataset = TensorDataset(input_ids, attention_mask)
batch_size = 8  # Adjust based on your available memory
dataloader = DataLoader(dataset, batch_size=batch_size)

# Set the model to evaluation mode
best_model.eval()

predictions = []

# Make predictions in batches
with torch.no_grad():
    for batch in dataloader:
        input_ids_batch, attention_mask_batch = [data.to(device) for data in batch]
        outputs = best_model(input_ids=input_ids_batch, attention_mask=attention_mask_batch)
        logits = outputs.logits
        predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())

In [ ]:
# Add predictions to your DataFrame
df_unique['quoted_pred'] = predictions

In [ ]:
df_unique.head()

In [ ]:
df_unique.quoted_pred.value_counts(dropna=False)

In [ ]:
# save the df to csv
df_unique.to_csv('actor_entities_extracted_fullNOS_quoted.csv', index = False, sep=';', quoting=csv.QUOTE_NONNUMERIC, encoding = 'utf-8')